In [3]:
import torch
import numpy as np
import random
__all__ = ['normalize_IQ', 'get_amp_phase', 'get_apf1', 'get_apf2', 'zero_mask', 'get_iq_framed']


def normalize_IQ(t):
    '''
    将IQ信号归一化到[0,1]区间

    t：(2,1024)
    '''
    t_max = np.max(t)
    t_min = np.min(t)
    diff = t_max - t_min
    t = (t - t_min) / diff
    return t


def get_amp_phase(data):
    '''
    将IQ信号转换为幅度-相位表示

    data：(2,1024)
    '''
    signal_len = data.shape[-1]

    # IQ信号 -> 复信号
    X_cmplx = data[0, :] + 1j * data[1, :]

    # 幅度
    X_amp = np.abs(X_cmplx)

    # 归一化包裹相位
    X_ang = np.arctan2(data[1, :], data[0, :]) / np.pi

    # X：(2, 1024)
    X = np.stack((X_amp, X_ang), axis=0)

    # 幅度：L2范数归一化
    # L2范数：平方和开根号
    X[0, :] = X[0, :] / np.linalg.norm(X[0, :], 2)

    return X


def get_apf1(data):  # 2*1024
    '''
    提取IQ信号的幅度-相位 -频率特征
    使用功率谱估计进行幅度归一化

    data：(2,1024)
    '''

    signal_len = data.shape[-1]

    # IQ信号 -> 复信号
    X_cmplx = data[0, :] + 1j * data[1, :]

    # 幅度
    X_amp = np.abs(X_cmplx)

    # 幅度处理 -> fft -> fftshift -> 幅度谱 -> 能量谱 -> 功率谱 -> 最大值gamma
    gamma = np.max(np.abs(np.fft.fftshift(np.fft.fft(X_amp/(np.mean(X_amp)-1), signal_len)))**2/signal_len)

    # 幅度归一化：基于功率谱估计
    X_amp = X_amp/gamma

    # 归一化包裹相位
    X_ang = np.arctan2(data[1, :], data[0, :]) / np.pi

    # 连续相位
    X_tmp = np.unwrap(np.angle(X_cmplx))

    # 归一化角频率
    X_freq = np.hstack((X_tmp[0], np.diff(X_tmp))) / np.pi

    X = np.stack((X_amp, X_ang, X_freq), axis=0)

    # X：(3, 1024)
    return X


def get_apf2(data):
    '''
    提取IQ信号的幅度-相位 -频率特征
    使用标准差进行幅度标准化

    data：(2,1024)
    '''

    signal_len = data.shape[-1]

    # IQ信号 -> 复信号
    X_cmplx = data[0, :] + 1j * data[1, :]

    # 幅度
    X_amp = np.abs(X_cmplx)

    # 幅度处理：减去均值再除以均值
    X_nc = X_amp/np.mean(X_amp)-1

    # 幅度处理后的标准差：均方值 - 均值的平方，再开方
    sigma = np.sqrt((X_nc**2).mean()-(np.abs(X_nc).mean())**2)

    # 幅度标准化：零均值单位方差
    X_amp = X_nc/sigma

    # 归一化包裹相位
    X_ang = np.arctan2(data[1, :], data[0, :]) / np.pi

    # 连续相位
    X_tmp = np.unwrap(np.angle(X_cmplx))

    # 归一化角频率
    # np.diff(X_tmp)：一阶差分长度减一
    # np.hstack()：左侧拼接原始数据X_tmp[0]
    # / np.pi：归一化
    X_freq = np.hstack((X_tmp[0], np.diff(X_tmp))) / np.pi

    X = np.stack((X_amp, X_ang, X_freq), axis=0)

    # X：(3, 1024)
    return X


def zero_mask(X_train, p=0.1):  # 2*1024
    '''
    零掩码处理 (zero_mask)：
    随机将一定比例的信号点置零
    模拟信号缺失或噪声干扰场景
    增强模型对不完整信号的鲁棒性
    '''

    num = int(X_train.shape[1] * p)
    res = X_train
    res[:, random.sample(range(X_train.shape[1]), num)] = 0
    return res


def get_iq_framed(data, L=32, R=16):
    '''
    IQ 帧提取 (get_iq_framed)：使用滑动窗口将长序列分割为短帧

    L: 帧长度
    R: 滑动步长
    每帧展平后堆叠形成新的特征矩阵
    适用于处理时序信号的局部特征
    '''
    # X：(2, 1024)
    X = normalize_IQ(data)

    Y = []

    for idx in range(0, X.shape[-1]-L+1, R):
        # (2, L=32) -> (64, )
        Y.append(X[:, idx:idx+L].reshape(-1))

    # Y：(F, 2L) = (63, 64)  F=(1024-L)/R+1
    Y = np.vstack(Y)

    return Y